In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 76.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=59f3f2ff03ea95ed4440bbd751f6c2a0a9235840ff5fe058ec20be459185559f
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
simulator = BasicSimulator()

# Custom random bit generator using a superposition state
def get_quantum_random_bit():
    qrng_qc = QuantumCircuit(1, 1)
    qrng_qc.h(0)
    qrng_qc.measure(0, 0)

    job = transpile(qrng_qc, simulator)
    result = simulator.run(job, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

In [3]:
NUM_QUBITS = 20

# Alice generates her bits and chooses bases (0=std, 1=diag)
alice_bits = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]
alice_bases = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]

qc = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)

# Encode the states
for i in range(NUM_QUBITS):
    if alice_bits[i] == 1:
        qc.x(i)
    if alice_bases[i] == 1:
        qc.h(i)

print("Alice has prepared and sent the qubits to Bob.")

Alice has prepared and sent the qubits to Bob.


In [4]:
# Bob chooses his random bases
bob_bases = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]

# Measure received qubits
for i in range(NUM_QUBITS):
    if bob_bases[i] == 1:
        qc.h(i)
    qc.measure(i, i)

job = transpile(qc, simulator)
result = simulator.run(job, shots=1).result()
counts = result.get_counts()

# Reverse the string to match indices
measured_str = list(counts.keys())[0][::-1]
bob_bits = [int(bit) for bit in measured_str]

print("Bob has finished measuring the received qubits.")

Bob has finished measuring the received qubits.


In [5]:
alice_key = []
bob_key = []

# Keep bits where bases match
for i in range(NUM_QUBITS):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_bits[i])

print(f"Alice bits:   {alice_bits}")
print(f"Alice bases:  {alice_bases}")
print(f"Bob bases:    {bob_bases}")
print(f"Bob bits:     {bob_bits}")
print("-" * 40)
print(f"Alice key:  {alice_key}")
print(f"Bob key:    {bob_key}")
print(f"Keys match: {alice_key == bob_key}")

Alice bits:   [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0]
Alice bases:  [1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
Bob bases:    [1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1]
Bob bits:     [1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1]
----------------------------------------
Alice key:  [1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1]
Bob key:    [1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1]
Keys match: True
